# AIMO 3 Submission - T4 GPU Version

Optimized for Tesla T4 (16GB VRAM). Uses 7B model with float16.

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# No vllm - use transformers instead (more compatible with Kaggle)
!pip install accelerate -q
print("Done")

In [ ]:
import os
import re
import subprocess
import tempfile
from collections import Counter
from dataclasses import dataclass
import pandas as pd
from tqdm import tqdm

IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

@dataclass
class Config:
    # 7B model - fits on T4
    model_id: str = "Qwen/Qwen2.5-Math-7B-Instruct"
    num_samples: int = 32
    num_generations: int = 4
    temperature: float = 0.7
    max_new_tokens: int = 2048
    timeout: int = 5

config = Config()
print(f"Model: {config.model_id}")

In [ ]:
# Code executor
class PythonREPL:
    def __init__(self, timeout=5):
        self.timeout = timeout
    
    def __call__(self, code):
        imports = "import math\nimport numpy as np\nimport sympy as sp\nfrom sympy import *\n"
        code = imports + code
        lines = code.strip().split("\n")
        if "print(" not in lines[-1] and "=" not in lines[-1]:
            lines[-1] = f"print({lines[-1]})"
        code = "\n".join(lines)
        
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(code)
            path = f.name
        try:
            result = subprocess.run(["python3", path], capture_output=True, text=True, timeout=self.timeout)
            os.unlink(path)
            return (True, result.stdout.strip()[:500]) if result.returncode == 0 else (False, result.stderr[-200:])
        except Exception as e:
            os.unlink(path)
            return False, str(e)

executor = PythonREPL(config.timeout)

In [ ]:
# Answer extraction
def extract_boxed(text):
    idx = text.rfind("\\boxed")
    if idx < 0: return None
    i, depth = idx, 0
    while i < len(text):
        if text[i] == "{": depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0: break
        i += 1
    return text[idx+7:i] if depth == 0 else None

def to_int(text):
    if not text: return -1
    text = re.sub(r"[\\$,\s{}]", "", text)
    try: return max(0, min(99999, round(float(text))))
    except: return -1

In [ ]:
# Load model using transformers (not vLLM)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {config.model_id}...")

tokenizer = AutoTokenizer.from_pretrained(config.model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Model loaded!")

In [ ]:
# Solver using transformers
def solve(problem):
    prompt = f"""Solve this math problem step by step. Use Python code when helpful.
Put your final answer in \\boxed{{}}.

Problem: {problem}

Solution:"""
    
    answers = []
    
    for _ in range(config.num_samples):
        current = prompt
        
        for _ in range(config.num_generations):
            inputs = tokenizer(current, return_tensors="pt").to(model.device)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=config.max_new_tokens,
                    temperature=config.temperature,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            
            text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Execute code if present
            if "```python" in text and text.rstrip().endswith("```"):
                blocks = re.findall(r"```python(.*?)```", text, re.DOTALL)
                if blocks:
                    _, out = executor(blocks[-1])
                    text = text + f"\n```output\n{out}\n```\n"
            
            current = text
            
            # Stop if answer found
            if "\\boxed" in text:
                break
        
        # Extract answer
        val = to_int(extract_boxed(current))
        if val >= 0:
            answers.append(val)
    
    if not answers:
        return 0
    return Counter(answers).most_common(1)[0][0]

print("Solver ready")

In [ ]:
# Main
if IS_SUBMISSION:
    import kaggle_evaluation.aimo_3_inference_server
    def predict(df):
        return pd.DataFrame({"id": df["id"], "answer": [solve(df["problem"].iloc[0])]})
    server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    server.serve()
else:
    ref = pd.read_csv("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv")
    correct = 0
    for i, row in ref.iterrows():
        pred = solve(row["problem"])
        ok = pred == row["answer"]
        correct += ok
        print(f"{i+1}. True={row['answer']}, Pred={pred} {'✓' if ok else '✗'}")
    print(f"\nAccuracy: {correct}/{len(ref)} = {correct/len(ref):.0%}")